In [ ]:
from pathlib import Path
from collections import defaultdict

import scanpy as sc
import pandas as pd
import os
import sys


In [ ]:
from preprocessing.scripts.data_loading import load_metadata_txt
from preprocessing.scripts.inspect_fingerprints import get_fingerprint_all, draw_molecules, analyze_fingerprint_collision_all
from preprocessing.scripts.data_preprocessing import add_fingerprints, del_false_duplicate_fp

In [ ]:
METADATA_EDITED_FOLDER = Path("./Metadata_edited/")
if not os.path.exists(METADATA_EDITED_FOLDER):
    os.mkdir(METADATA_EDITED_FOLDER)

In [ ]:
merged_LINCS_metadata_dir = Path("/home/ani/Documents/uni/prnet_eval/dataset/metadata/LINCS")


In [ ]:
comp_info_merged = load_metadata_txt(merged_LINCS_metadata_dir / "compoundinfo_beta.txt")
gene_info_merged = load_metadata_txt(merged_LINCS_metadata_dir / "geneinfo_beta.txt")
inst_info_merged = load_metadata_txt(merged_LINCS_metadata_dir / "instinfo_beta.txt")

In [ ]:
comp_info_merged = comp_info_merged[
    comp_info_merged['canonical_smiles'].notna() &
    comp_info_merged['canonical_smiles'].astype(str).str.strip().str.lower().ne('restricted')
].reset_index(drop=True)

In [ ]:
leakage_records = []

for fold in range(5):
    train_fold = comp_info_merged[comp_info_merged[f'canonical_smiles_split_{fold}'] == 'train']
    test_fold = comp_info_merged[comp_info_merged[f'canonical_smiles_split_{fold}'] == 'test']

    train_compounds = set(train_fold['fingerprint_smiles'].dropna())
    test_compounds = set(test_fold['fingerprint_smiles'].dropna())

    train_compounds.discard('CONTROL')
    test_compounds.discard('CONTROL')

    overlap_traintest = train_compounds.intersection(test_compounds)

    train_leak_samples = train_fold['fingerprint_smiles'].isin(overlap_traintest).sum()
    test_leak_samples = test_fold['fingerprint_smiles'].isin(overlap_traintest).sum()
    total_fold_leak_samples = train_leak_samples + test_leak_samples

    leaked_smiles_mask = train_fold['fingerprint_smiles'].isin(overlap_traintest)
    train_test_smiles = sorted(train_fold.loc[leaked_smiles_mask, 'canonical_smiles'].unique().tolist())

    if len(overlap_traintest) > 0:
        print(f"FOLD {fold} LEAKAGE DETECTED:")
        print(f"  - Unique Leaked Compounds: {len(overlap_traintest)}")
        print(f"  - Affected Samples in Fold: {total_fold_leak_samples} ({train_leak_samples} train, {test_leak_samples} test)")
        print(f"  - Leaked SMILES: {train_test_smiles}\n")
    else:
        print(f"Fold {fold}: No compound leakage detected between train and test splits.")

    leakage_records.append({
        'fold': fold,
        'num_unique_leaked_compounds': len(overlap_traintest),
        'num_leaked_train_samples': train_leak_samples,
        'num_leaked_test_samples': test_leak_samples,
        'total_leaked_samples_in_fold': total_fold_leak_samples,
        'train_test_leaked_canonical_smiles': "; ".join(train_test_smiles),
    })

leakage_df = pd.DataFrame(leakage_records)
leakage_df.to_csv("fingerprint_leakage_smiles_split.csv", index=False)

Running analysis for: morgan


[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerat

Running analysis for: map4
Running analysis for: map4c
Running analysis for: erg
Running analysis for: physicochemical
Running analysis for: topological_torsion


Fingerprint Method,erg,map4,map4c,morgan,physicochemical,topological_torsion
Collision Reason,,,,,,
Bioisosteres (Same Skeleton),104,0,0,162,197,16
Compositional Isomer (Different Formula),995,1,1,234,1490,13
Defined vs Undefined Stereo,248,29,9,262,252,266
Dimer/Salt Variation,1,0,0,22,35,1
Stereoisomers (R/S conflict),3313,3950,2,4000,3132,4065
Structural Isomers (Positional),152,0,0,54,77,23
True duplicates,49,49,54,70,53,73
